# Phase 7 — Metadata Features + BERT+SVM Ensemble

Προσθήκη metadata features (country, year, month, day) στο SVM.

**Στρατηγική:**
- country → One-hot encoding
- year, month, day → Normalized αριθμητικά features
- Συνδυασμός: `hstack([TF-IDF matrix, metadata matrix])`
- Ξαναδοκιμάζουμε το probability ensemble με το BERT

**Αναμενόμενο baseline:** BERT+SVM Ensemble = 0.8028 (validation), 0.755 (Kaggle)

In [1]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score
import re
import warnings
warnings.filterwarnings('ignore')

In [2]:
train = pd.read_csv('train.csv')
valid = pd.read_csv('valid.csv')
test  = pd.read_csv('test.csv')

# Φόρτωση BERT predictions και probabilities (από phase4)
bert_valid = pd.read_csv('bert_valid_predictions.csv')
bert_test  = pd.read_csv('bert_test_predictions.csv')

bert_valid_hazard_probs  = np.load('npy/bert_valid_hazard_probs.npy')
bert_valid_product_probs = np.load('npy/bert_valid_product_probs.npy')
bert_test_hazard_probs   = np.load('npy/bert_test_hazard_probs.npy')
bert_test_product_probs  = np.load('npy/bert_test_product_probs.npy')

y_valid_hazard  = valid['hazard-category']
y_valid_product = valid['product-category']

print(f'Train: {len(train)} | Valid: {len(valid)} | Test: {len(test)}')
print(f'BERT valid hazard probs:  {bert_valid_hazard_probs.shape}')
print(f'BERT valid product probs: {bert_valid_product_probs.shape}')

Train: 5082 | Valid: 565 | Test: 997
BERT valid hazard probs:  (565, 10)
BERT valid product probs: (565, 22)


In [3]:
# Ανάλυση metadata
print('=== ΑΝΑΛΥΣΗ METADATA ===')
print(f'Unique countries (train): {train["country"].nunique()}')
print(f'Unique countries (all):   {pd.concat([train["country"], valid["country"], test["country"]]).nunique()}')
print(f'Year range:  {train["year"].min()} - {train["year"].max()}')
print(f'Month range: {train["month"].min()} - {train["month"].max()}')
print(f'Day range:   {train["day"].min()} - {train["day"].max()}')
print(f'\nMissing values:')
for col in ['country', 'year', 'month', 'day']:
    n = train[col].isna().sum()
    if n > 0:
        print(f'  {col}: {n} missing')
    else:
        print(f'  {col}: OK')

=== ΑΝΑΛΥΣΗ METADATA ===
Unique countries (train): 9
Unique countries (all):   9
Year range:  1994 - 2022
Month range: 1 - 12
Day range:   1 - 31

Missing values:
  country: OK
  year: OK
  month: OK
  day: OK


In [4]:
def preprocess_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def official_st1_score(y_true_hazard, y_pred_hazard,
                       y_true_product, y_pred_product, verbose=True):
    f1_hazard = f1_score(y_true_hazard, y_pred_hazard, average='macro', zero_division=0)
    y_true_hazard  = np.array(y_true_hazard)
    y_pred_hazard  = np.array(y_pred_hazard)
    y_true_product = np.array(y_true_product)
    y_pred_product = np.array(y_pred_product)
    mask = (y_true_hazard == y_pred_hazard)
    f1_product = f1_score(
        y_true_product[mask], y_pred_product[mask],
        average='macro', zero_division=0
    ) if mask.sum() > 0 else 0.0
    score = (f1_hazard + f1_product) / 2
    if verbose:
        print(f'macro-F1 Hazard:                    {f1_hazard:.4f}')
        print(f'Σωστά hazard predictions:           {mask.sum()}/{len(mask)} ({100*mask.mean():.1f}%)')
        print(f'macro-F1 Product (given correct h): {f1_product:.4f}')
        print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
        print(f'OFFICIAL ST1 SCORE:                 {score:.4f}')
    return score

In [5]:
# === ΒΗΜΑ 1: TF-IDF features (ίδια με phase4) ===
train['combined'] = train['title'].fillna('') + ' ' + train['text'].fillna('').str[:550]
valid['combined'] = valid['title'].fillna('') + ' ' + valid['text'].fillna('').str[:550]
test['combined']  = test['title'].fillna('')  + ' ' + test['text'].fillna('').str[:550]

train['combined'] = train['combined'].apply(preprocess_text)
valid['combined'] = valid['combined'].apply(preprocess_text)
test['combined']  = test['combined'].apply(preprocess_text)

tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    stop_words='english'
)

X_train_tfidf = tfidf.fit_transform(train['combined'])
X_valid_tfidf = tfidf.transform(valid['combined'])
X_test_tfidf  = tfidf.transform(test['combined'])

print(f'TF-IDF shape: {X_train_tfidf.shape}')

TF-IDF shape: (5082, 50000)


In [6]:
# === ΒΗΜΑ 2: Metadata features ===

# 2a. Country → One-hot encoding
# Fit σε train+valid+test για να δει όλες τις χώρες
all_countries = pd.concat([train[['country']], valid[['country']], test[['country']]])
all_countries['country'] = all_countries['country'].fillna('unknown')

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
ohe.fit(all_countries[['country']])

country_train = ohe.transform(train[['country']].fillna('unknown'))
country_valid = ohe.transform(valid[['country']].fillna('unknown'))
country_test  = ohe.transform(test[['country']].fillna('unknown'))

print(f'Country one-hot shape: {country_train.shape}  ({country_train.shape[1]} χώρες)')

# 2b. Year, Month, Day → Normalized [0,1]
# Fit scaler σε train μόνο
num_cols = ['year', 'month', 'day']

train_num = train[num_cols].fillna(0).values
valid_num = valid[num_cols].fillna(0).values
test_num  = test[num_cols].fillna(0).values

scaler = MinMaxScaler()
scaler.fit(train_num)

num_train = sp.csr_matrix(scaler.transform(train_num))
num_valid = sp.csr_matrix(scaler.transform(valid_num))
num_test  = sp.csr_matrix(scaler.transform(test_num))

print(f'Numeric features shape: {num_train.shape}')

# 2c. Συνδυασμός: TF-IDF + country_ohe + numeric
X_train = sp.hstack([X_train_tfidf, country_train, num_train])
X_valid = sp.hstack([X_valid_tfidf, country_valid, num_valid])
X_test  = sp.hstack([X_test_tfidf,  country_test,  num_test])

print(f'\nΤελικό feature matrix:')
print(f'  Train: {X_train.shape}')
print(f'  Valid: {X_valid.shape}')
print(f'  Test:  {X_test.shape}')
print(f'  (TF-IDF: 50000 + country OHE: {country_train.shape[1]} + numeric: 3)')

Country one-hot shape: (5082, 9)  (9 χώρες)
Numeric features shape: (5082, 3)

Τελικό feature matrix:
  Train: (5082, 50012)
  Valid: (565, 50012)
  Test:  (997, 50012)
  (TF-IDF: 50000 + country OHE: 9 + numeric: 3)


In [7]:
# === ΒΗΜΑ 3: SVM με metadata features ===
clf_svm_hazard = LinearSVC(C=0.5, class_weight='balanced', max_iter=2000, random_state=42)
clf_svm_hazard.fit(X_train, train['hazard-category'])

clf_svm_product = LinearSVC(C=0.5, class_weight='balanced', max_iter=2000, random_state=42)
clf_svm_product.fit(X_train, train['product-category'])

svm_valid_hazard  = clf_svm_hazard.predict(X_valid)
svm_valid_product = clf_svm_product.predict(X_valid)

print('=== SVM με Metadata Features ===')
score_svm_meta = official_st1_score(
    y_valid_hazard, svm_valid_hazard,
    y_valid_product, svm_valid_product
)
print(f'\nΣύγκριση SVM:')
print(f'  SVM χωρίς metadata: 0.7419')
print(f'  SVM με metadata:    {score_svm_meta:.4f}')

=== SVM με Metadata Features ===
macro-F1 Hazard:                    0.8004
Σωστά hazard predictions:           524/565 (92.7%)
macro-F1 Product (given correct h): 0.6818
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OFFICIAL ST1 SCORE:                 0.7411

Σύγκριση SVM:
  SVM χωρίς metadata: 0.7419
  SVM με metadata:    0.7411


In [8]:
# === ΒΗΜΑ 4: Probability Ensemble SVM+metadata + BERT ===
def probability_ensemble(svm_scores, bert_probs, svm_classes, w_svm, w_bert):
    scaler = MinMaxScaler()
    svm_norm = scaler.fit_transform(svm_scores)
    combined = w_svm * svm_norm + w_bert * bert_probs
    pred_indices = combined.argmax(axis=1)
    return svm_classes[pred_indices]

# Decision scores από SVM με metadata
svm_hazard_scores  = clf_svm_hazard.decision_function(X_valid)
svm_product_scores = clf_svm_product.decision_function(X_valid)

weight_combinations = [(0.9,0.1),(0.7,0.3),(0.5,0.5),(0.3,0.7),(0.1,0.9)]

print('=== PROBABILITY ENSEMBLE (SVM+Metadata) + BERT ===')
print('(Βάρη: SVM, BERT)\n')

best_score   = 0
best_weights = None

for w_svm, w_bert in weight_combinations:
    ens_hazard = probability_ensemble(
        svm_hazard_scores, bert_valid_hazard_probs,
        clf_svm_hazard.classes_, w_svm, w_bert
    )
    ens_product = probability_ensemble(
        svm_product_scores, bert_valid_product_probs,
        clf_svm_product.classes_, w_svm, w_bert
    )
    score = official_st1_score(
        y_valid_hazard, ens_hazard,
        y_valid_product, ens_product,
        verbose=False
    )
    print(f'SVM={w_svm}, BERT={w_bert} → Score: {score:.4f}')
    if score > best_score:
        best_score   = score
        best_weights = (w_svm, w_bert)

print(f'\nΒέλτιστα βάρη: SVM={best_weights[0]}, BERT={best_weights[1]}')
print(f'\n=== ΣΥΓΚΡΙΣΗ ===')
print(f'BERT+SVM (χωρίς metadata):  0.8028  (Kaggle: 0.755) ← previous best')
print(f'BERT+SVM (με metadata):     {best_score:.4f}')

=== PROBABILITY ENSEMBLE (SVM+Metadata) + BERT ===
(Βάρη: SVM, BERT)

SVM=0.9, BERT=0.1 → Score: 0.6588
SVM=0.7, BERT=0.3 → Score: 0.7394
SVM=0.5, BERT=0.5 → Score: 0.7876
SVM=0.3, BERT=0.7 → Score: 0.8028
SVM=0.1, BERT=0.9 → Score: 0.7998

Βέλτιστα βάρη: SVM=0.3, BERT=0.7

=== ΣΥΓΚΡΙΣΗ ===
BERT+SVM (χωρίς metadata):  0.8028  (Kaggle: 0.755) ← previous best
BERT+SVM (με metadata):     0.8028


In [9]:
# === ΒΗΜΑ 5: Submission με τα βέλτιστα βάρη ===
w_svm_best, w_bert_best = best_weights

svm_test_hazard_scores  = clf_svm_hazard.decision_function(X_test)
svm_test_product_scores = clf_svm_product.decision_function(X_test)

ensemble_test_hazard = probability_ensemble(
    svm_test_hazard_scores, bert_test_hazard_probs,
    clf_svm_hazard.classes_, w_svm_best, w_bert_best
)
ensemble_test_product = probability_ensemble(
    svm_test_product_scores, bert_test_product_probs,
    clf_svm_product.classes_, w_svm_best, w_bert_best
)

submission = pd.DataFrame({
    'id': test['id'],
    'hazard-category': ensemble_test_hazard,
    'product-category': ensemble_test_product
})
submission.to_csv('submission_metadata_ensemble.csv', index=False)
print('Αποθηκεύτηκε: submission_metadata_ensemble.csv')
print(f'Αριθμός predictions: {len(submission)}')
print('\n--- Πρώτες 5 προβλέψεις ---')
print(submission.head())

Αποθηκεύτηκε: submission_metadata_ensemble.csv
Αριθμός predictions: 997

--- Πρώτες 5 προβλέψεις ---
   id hazard-category              product-category
0   0      biological  meat, egg and dairy products
1   1      biological  meat, egg and dairy products
2   2      biological    prepared dishes and snacks
3   3      biological  meat, egg and dairy products
4   4  foreign bodies             ices and desserts


In [10]:
# Έλεγξε αν τα metadata σχετίζονται με τις κατηγορίες
print('=== TOP COUNTRIES ανά hazard-category ===')
print(train.groupby('hazard-category')['country'].agg(lambda x: x.value_counts().index[0]))

print('\n=== Κατανομή hazard ανά year ===')
print(pd.crosstab(train['year'], train['hazard-category']))

=== TOP COUNTRIES ανά hazard-category ===
hazard-category
allergens                         us
biological                        us
chemical                          us
food additives and flavourings    au
foreign bodies                    us
fraud                             us
migration                         ie
organoleptic aspects              au
other hazard                      us
packaging defect                  au
Name: country, dtype: object

=== Κατανομή hazard ανά year ===
hazard-category  allergens  biological  chemical  \
year                                               
1994                     0           5         1   
1995                     0           5         1   
1996                     0           3         0   
1997                     0           7         1   
1998                     3           7         1   
1999                     6           9         3   
2000                     1           6         1   
2001                     2          13   